# kronos-semi · Notebook 05: Axisymmetric 2D MOSCAP — LF and HF C–V

This notebook extends the planar 2D MOS capacitor (Notebook 03) to an
**axisymmetric cylindrical** geometry and reproduces the classic
**low-frequency vs. high-frequency C–V split** shown in
Hu, *Modern Semiconductor Devices for Integrated Circuits*, Fig. 5-18.

### Device

| Parameter | Value |
|-----------|-------|
| Substrate | P-Si, $N_A = 10^{17}$ cm⁻³ |
| Oxide     | SiO₂, $t_{ox} = 5$ nm |
| Gate radius | $R_{gate} = 5$ μm |
| Gate work function | 4.05 eV (N⁺ poly) |

### LF vs HF physics

- **Low-frequency (quasi-static)**: The AC signal is slow enough that minority carriers
  (electrons in the P-body) can follow the gate voltage. Inversion charge builds up
  and the capacitance returns to $C_{ox}$ in strong inversion.
- **High-frequency**: Minority carriers cannot follow the fast AC signal; only the
  depletion edge responds. The capacitance saturates at
  $C_{min} = C_{ox} C_{dmax} / (C_{ox} + C_{dmax})$ in inversion.

### Simulation approach

Both modes solve the **axisymmetric equilibrium Poisson equation** in $(r,z)$
with an $r$-weighted measure $r\,dr\,dz$ (the $2\pi$ azimuthal factor cancels
in the normalised capacitance $C/C_{ox}$). The semiconductor charge
$$Q_{sub}(V_g) = 2\pi\int_{Si} \rho(r,z;V_g)\,r\,dr\,dz$$
is differentiated numerically to give the capacitance:
$$C(V_g) = -\frac{Q_{sub}(V_g+\delta V)-Q_{sub}(V_g-\delta V)}{2\delta V A_{gate}}$$

For HF, the minority carrier density $n_{hat}$ is **frozen** at its DC value
before the $\pm\delta V$ perturbation solves, so only majority holes respond.

## Notebook set

| Notebook | Covers |
|----------|--------|
| [01](./01_pn_junction_1d.ipynb) | Equilibrium Poisson, 1D pn junction |
| [02](./02_pn_junction_bias.ipynb) | Bias sweep, Slotboom DD, SRH |
| [03](./03_mos_cv.ipynb) | 2D MOS C-V, multi-region |
| [04](./04_resistor_3d.ipynb) | 3D doped resistor V-I |
| **05 — Axisymmetric MOS C-V** (this one) | LF vs HF C-V split, Hu Fig. 5-18 |

## 1. Install dolfinx on Colab

Skip this cell if dolfinx is already installed (e.g. in the Docker dev-container).

In [ ]:
import importlib, subprocess, sys

def _colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _colab():
    print('Installing dolfinx via fem-on-colab …')
    subprocess.run(
        ['wget', '-q',
         'https://github.com/fem-on-colab/fem-on-colab.github.io/raw/gh-pages/releases/fenics-install-real.sh',
         '-O', '/tmp/fenics-install.sh'],
        check=True
    )
    subprocess.run(['bash', '/tmp/fenics-install.sh'], check=True)
    # Clone or update the repo
    import os
    if not os.path.isdir('/content/kronos-semi'):
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/rwalkerlewis/kronos-semi',
             '/content/kronos-semi'],
            check=True
        )
    sys.path.insert(0, '/content/kronos-semi')
else:
    print('Not on Colab — assuming dolfinx is already installed.')

## 2. Analytical reference values (Hu Ch. 5)

Compute $V_{FB}$, $V_T$, $W_{dmax}$, $C_{min}/C_{ox}$ from closed-form expressions.

In [ ]:
import math
import numpy as np

# Physical constants
Q_e   = 1.602176634e-19   # C
KB    = 1.380649e-23      # J/K
EPS0  = 8.8541878128e-12  # F/m
T     = 300.0             # K
Vt    = KB * T / Q_e      # thermal voltage, ~0.02585 V

# Device parameters
eps_r_Si  = 11.7
eps_r_ox  = 3.9
t_ox      = 5.0e-9        # 5 nm
N_A       = 1.0e17 * 1e6  # m^-3
n_i       = 1.0e10 * 1e6  # m^-3
chi_Si    = 4.05          # eV  (Si electron affinity)
Eg_Si     = 1.12          # eV
phi_m_npoly = 4.05        # eV  (N+ poly work function ≈ chi_Si)
R_gate    = 5.0e-6        # m

# Derived
eps_Si  = eps_r_Si * EPS0
eps_ox  = eps_r_ox * EPS0
Cox     = eps_ox / t_ox           # F/m^2
phi_B   = Vt * math.log(N_A / n_i)  # bulk potential
phi_si  = chi_Si + Eg_Si/2 - phi_B   # work function of P-Si
phi_ms  = phi_m_npoly - phi_si
Vfb     = phi_ms                  # ignoring fixed oxide charge
Wdmax   = math.sqrt(2 * eps_Si * 2 * phi_B / (Q_e * N_A))
Cdmax   = eps_Si / Wdmax
Cmin    = Cox * Cdmax / (Cox + Cdmax)
Vt_thr  = Vfb + 2*phi_B + Q_e * N_A * Wdmax / Cox  # threshold

print(f'Vt (thermal)      = {Vt*1000:.3f} mV')
print(f'Cox               = {Cox*1e-4:.4e} F/cm²')
print(f'phi_B             = {phi_B:.4f} V')
print(f'Vfb               = {Vfb:.4f} V')
print(f'Wdmax             = {Wdmax*1e9:.2f} nm')
print(f'Cmin/Cox          = {Cmin/Cox:.4f}')
print(f'Threshold Vt      = {Vt_thr:.4f} V')

## 3. Load config and build mesh

We use a coarser mesh than the benchmark (`r=10, z_si=40, z_oxide=8`) so the sweep finishes in a few minutes on Colab.

In [ ]:
import json, pathlib

# Locate repo root whether running in-repo or on Colab
import os, sys
for candidate in ['.', '/content/kronos-semi']:
    if pathlib.Path(candidate, 'semi').is_dir():
        sys.path.insert(0, str(pathlib.Path(candidate).resolve()))
        break

import semi.schema as schema

# Coarse mesh for fast notebook run; increase resolution for production
cfg = schema.validate({
    'schema_version': '1.2.0',
    'name': 'moscap_axi_nb',
    'dimension': 'axisymmetric_2d',
    'mesh': {
        'source': 'builtin_axi',
        'extents': {'r': [0.0, 10.0e-6], 'z': [0.0, 1.005e-6]},
        'layers': [
            {'name': 'oxide',   'material': 'SiO2', 'z_range': [1.0e-6, 1.005e-6], 'tag': 2},
            {'name': 'silicon', 'material': 'Si',   'z_range': [0.0,   1.0e-6],    'tag': 1}
        ],
        'resolution': {'r': 10, 'z_oxide': 8, 'z_si': 40},
        'facets': [
            {'name': 'axis',        'tag': 10, 'where': 'r=0'},
            {'name': 'outer',       'tag': 11, 'where': 'r=R_outer'},
            {'name': 'body',        'tag': 12, 'where': 'z=z_bot'},
            {'name': 'gate',        'tag': 13, 'where': 'z=z_top,r<=R_gate'},
            {'name': 'field_oxide', 'tag': 14, 'where': 'z=z_top,r>R_gate'}
        ],
        'R_gate': 5.0e-6
    },
    'regions': {
        'silicon': {'material': 'Si',   'tag': 1, 'role': 'semiconductor'},
        'oxide':   {'material': 'SiO2', 'tag': 2, 'role': 'insulator'}
    },
    'doping': [
        {'region': 'silicon', 'profile': {'type': 'uniform', 'N_D': 0.0, 'N_A': 1.0e17}}
    ],
    'contacts': [
        {'name': 'body', 'facet': 'body', 'type': 'ohmic',    'voltage': 0.0},
        {'name': 'gate', 'facet': 'gate', 'type': 'mos_gate',
         'work_function_eV': 4.05, 'voltage': 0.0}
    ],
    'physics': {
        'temperature': 300.0,
        'statistics': 'boltzmann',
        'mobility': {'mu_n': 1400.0, 'mu_p': 450.0},
        'recombination': {'srh': False, 'tau_n': 1e-7, 'tau_p': 1e-7, 'E_t': 0.0}
    },
    'solver': {'type': 'cv_lf'}
})
print('Config loaded and validated.')

## 4. Run LF and HF C–V sweeps

In [ ]:
from semi.physics.cv import compute_cv_curve

# Bias sweep — fewer points for speed (increase Vg_step for even faster runs)
Vg_sweep = list(np.arange(-1.5, 1.55, 0.1))
dV = 0.005

print('Running LF sweep …')
res_lf = compute_cv_curve(cfg, mode='LF', Vg_sweep=Vg_sweep, dV=dV)
print(f'  Done: {len(res_lf["Vg"])} bias points.')

print('Running HF sweep …')
res_hf = compute_cv_curve(cfg, mode='HF', Vg_sweep=Vg_sweep, dV=dV)
print(f'  Done: {len(res_hf["Vg"])} bias points.')

## 5. Plot C/Cox vs Vg — Hu Fig. 5-18 style overlay

The plot should show:
1. **Accumulation** (Vg ≪ Vfb): both curves at ~Cox
2. **Flat band** (Vg = Vfb): both drop below Cox
3. **Depletion**: both curves coincide and decrease
4. **Inversion** (Vg > Vt): LF rises back to Cox; HF saturates at Cmin

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

Vg_lf = np.array(res_lf['Vg'])
C_lf  = np.array(res_lf['C'])    # F/m^2
Vg_hf = np.array(res_hf['Vg'])
C_hf  = np.array(res_hf['C'])    # F/m^2

fig, ax = plt.subplots(figsize=(8, 5))

# Simulated curves
ax.plot(Vg_lf, C_lf / Cox, 'b-o', ms=4, label='LF (quasi-static) — simulation')
ax.plot(Vg_hf, C_hf / Cox, 'r-s', ms=4, label='HF (frozen minority) — simulation')

# Reference horizontal lines
ax.axhline(1.0,           color='gray', lw=1, ls='--', label=f'$C_{{ox}}$')
ax.axhline(Cmin / Cox,    color='darkorange', lw=1, ls=':', label=f'$C_{{min}}/C_{{ox}} = {Cmin/Cox:.3f}$ (analytical)')

# Reference vertical lines
ax.axvline(Vfb,    color='purple', lw=1, ls='-.', label=f'$V_{{FB}} = {Vfb:.3f}$ V')
ax.axvline(Vt_thr, color='green',  lw=1, ls='-.', label=f'$V_T = {Vt_thr:.3f}$ V')

# Annotations for four regions
ax.text(-1.4, 1.02, 'Accum.', fontsize=9, color='navy')
ax.text(Vfb + 0.05, 0.85, 'Depletion', fontsize=9, color='navy')
ax.text(Vt_thr + 0.05, 0.5, 'Inversion', fontsize=9, color='navy')

ax.set_xlabel('Gate voltage $V_g$ (V)', fontsize=12)
ax.set_ylabel('$C / C_{ox}$', fontsize=12)
ax.set_title(
    r'Axisymmetric 2D MOSCAP — LF and HF C–V'
    '\n'
    r'$N_A = 10^{17}$ cm⁻³, $t_{ox} = 5$ nm, $R_{gate} = 5$ μm'
    , fontsize=11
)
ax.set_ylim(0.0, 1.15)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('moscap_axi_cv.png', dpi=150)
plt.show()
print('Figure saved to moscap_axi_cv.png')

## 6. Quantitative checks

Compare simulation against analytical predictions from Hu §§5.1–5.6.

In [ ]:
# Accumulation check: C_LF at most negative Vg should be close to Cox
acc_C_lf = C_lf[0] / Cox
print(f'C_LF/Cox at Vg = {Vg_lf[0]:.2f} V (accumulation): {acc_C_lf:.3f}  (expect > 0.90)')

# Inversion check: C_LF at most positive Vg should return toward Cox
inv_C_lf = C_lf[-1] / Cox
print(f'C_LF/Cox at Vg = {Vg_lf[-1]:.2f} V (inversion):   {inv_C_lf:.3f}  (expect > 0.70)')

# HF inversion: C_HF should be well below C_LF
inv_C_hf = C_hf[-1] / Cox
print(f'C_HF/Cox at Vg = {Vg_hf[-1]:.2f} V (HF inversion): {inv_C_hf:.3f}')
print(f'Analytical Cmin/Cox:                              {Cmin/Cox:.3f}')

# LF/HF agreement in mid-depletion
dep_mask = (Vg_lf > Vfb) & (Vg_lf < 0.0)
if dep_mask.any():
    ratio = C_lf[dep_mask] / C_hf[np.where(dep_mask)[0]]
    print(f'LF/HF ratio in depletion: {ratio.min():.3f} – {ratio.max():.3f}  (expect ~1)')

print('\nAll qualitative checks done.')

## 7. Discussion

### Four regions of the C–V curve

**Accumulation** ($V_g \ll V_{FB}$): negative gate voltage attracts
holes to the Si surface. The hole layer screens the gate and both curves
asymptote to $C_{ox}$ (capacitor fully charged by majority carriers).

**Flat-band** ($V_g = V_{FB} \approx -0.98$ V for this device): the
surface charge is zero. $C_{FB} < C_{ox}$ because the Debye tail in the
semiconductor adds in series.

**Depletion** ($V_{FB} < V_g < V_T$): ionised acceptors form a
space-charge layer of width $W_{dep}$. Both LF and HF curves coincide
here — the depletion edge adjusts through majority carrier redistribution,
which is fast at both low and high frequency. $C = C_{ox}C_d/(C_{ox}+C_d)$
with $C_d = \varepsilon_{Si}/W_{dep}$ decreasing as $W_{dep}$ grows.

**Inversion** ($V_g > V_T \approx 0.10$ V):

- *LF*: the ohmic body contact supplies minority electrons on the
  quasi-static timescale. Electrons accumulate at the surface, the
  depletion width stops growing, and $C_{LF}$ rises back toward $C_{ox}$.
- *HF*: the AC signal is faster than the minority generation/recombination
  time $\tau_{gen}$. The inversion layer charge cannot follow the signal;
  only the depletion edge moves. $C_{HF}$ saturates at
  $C_{min} = C_{ox} C_{dmax}/(C_{ox}+C_{dmax})$.

The analytical $C_{min}/C_{ox}$ for these parameters is
approximately **0.19–0.22** (depends on precise $V_{FB}$ correction).

### Axisymmetric formulation

The weak form integrates over the 2D $(r,z)$ cross-section with the
Jacobian factor $r$:
$$\int\int \varepsilon \nabla\psi \cdot \nabla v \, r \, dr\,dz = \cdots$$
This is equivalent to the full 3D form after integrating out the
azimuthal angle. The $r=0$ axis carries a natural homogeneous Neumann
condition: the $r$ factor in the measure removes the apparent
singularity at the axis.